<a href="https://colab.research.google.com/github/Kintsukuro1/Mineria-Datos/blob/main/proyecto_mineria_felipe_ruiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto Base - Evaluación de Minería de Datos

## Información del Grupo

- **Nombre o Número de Grupo:** Grupo 1
- **Sección:** BIY7121-004D
- **Fecha de Entrega:** [Indicar la fecha de entrega del informe]

---

## Integrantes del Grupo

### Estudiante 1
- **Nombre:** Felipe
- **Apellido:** Ruiz
- **Correo institucional:** fe.ruizm@duocuc.cl

---

## Tema del Proyecto
Análisis y predicción del comportamiento de juegos de casino online a partir de variables como RTP, volatilidad, multiplicadores y características de los juegos.

> Clasificación de juegos de casino online según su potencial de retorno y multiplicador máximo, en base a variables técnicas y comerciales del dataset.

---

## Descripción del Proyecto
Este proyecto analiza un dataset de juegos de casino online con el fin de identificar patrones en las características de los juegos que permitan segmentarlos por atractivo para el jugador. El modelo resultante podría ser utilizado por operadores de casinos para personalizar su oferta y optimizar la retención de usuarios.

---

## Introducción a la Metodología

Este documento sigue la estructura de la metodología **CRISP-DM** (*Cross Industry Standard Process for Data Mining*), compuesta por seis fases fundamentales que guían el análisis de datos desde la comprensión del negocio hasta la evaluación del modelo.

---


## 1. Comprensión del Negocio

El objetivo es analizar un dataset de juegos de casino online para identificar patrones que ayuden a mejorar la retención de usuarios, optimizar decisiones comerciales y preparar una base sólida para modelado predictivo.

Una empresa de juegos de azar online puede utilizar este modelo para:
- Detectar qué juegos combinan mejor retorno al jugador (RTP) con multiplicadores altos.
- Personalizar recomendaciones según el perfil de riesgo del usuario.
- Optimizar su catálogo de juegos para maximizar engagement y rentabilidad.

**Decisión esperada:** clasificar juegos como "alto potencial" vs "estándar" para priorizar su promoción.


In [ ]:
# No requiere código en esta fase

## 2. Comprensión de los Datos

- **Fuente del dataset:** `online_casino_games_dataset_v2.csv`
- **Variables objetivo:** `rtp` (retorno al jugador) y `max_multiplier` (multiplicador máximo)
- **Variables predictoras:** casino, proveedor, tipo de juego, volatilidad, jackpot, apuesta mínima, ganancia máxima, compatibilidad móvil, free spins, compra de bonus, año de lanzamiento, jurisdicción, moneda.
- **Tipos de datos:** numéricos continuos, categóricos nominales/ordinales, booleanos, temporales.

A continuación se carga el dataset y se inspecciona su estructura inicial.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar dataset
df = pd.read_csv('/content/online_casino_games_dataset_v2.csv')

print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head()


In [ ]:
# Inspección estructural: tipos de datos y conteo de no-nulos
df.info()


In [ ]:
# Nombres de columnas disponibles
print(df.columns.tolist())


In [ ]:
# Diccionario de datos: mapeo de columnas
diccionario_columnas = {
    "casino":               ("casino",                  "Nombre del casino",                        "Categórica", "Nominal"),
    "game":                 ("juego",                   "Nombre del juego",                         "Categórica", "Nominal"),
    "provider":             ("proveedor",               "Proveedor del juego",                      "Categórica", "Nominal"),
    "rtp":                  ("retorno_al_jugador",      "Porcentaje de retorno esperado",            "Numérica",   "Continua"),
    "volatility":           ("volatilidad",             "Nivel de volatilidad del juego",            "Categórica", "Ordinal"),
    "jackpot":              ("tiene_jackpot",           "Si el juego incluye jackpot",               "Categórica", "Binaria"),
    "country_availability": ("paises_disponibles",      "Países donde está habilitado",              "Categórica", "Nominal"),
    "min_bet":              ("apuesta_minima",          "Monto mínimo para apostar",                 "Numérica",   "Continua"),
    "max_win":              ("ganancia_maxima",         "Ganancia máxima potencial",                 "Numérica",   "Continua"),
    "game_type":            ("tipo_juego",              "Tipo general del juego",                    "Categórica", "Nominal"),
    "game_category":        ("categoria_juego",         "Categoría específica del juego",            "Categórica", "Nominal"),
    "license_jurisdiction": ("jurisdiccion_licencia",   "Jurisdicción regulatoria",                  "Categórica", "Nominal"),
    "release_year":         ("anio_lanzamiento",        "Año de lanzamiento",                        "Numérica",   "Discreta"),
    "currency":             ("moneda",                  "Moneda principal del juego",                "Categórica", "Nominal"),
    "mobile_compatible":    ("compatible_movil",        "Compatibilidad con móviles",                "Categórica", "Binaria"),
    "free_spins_feature":   ("tiene_tiros_gratis",      "Si tiene función de free spins",            "Categórica", "Binaria"),
    "bonus_buy_available":  ("compra_bonus_disponible", "Permite comprar bono directamente",         "Categórica", "Binaria"),
    "max_multiplier":       ("multiplicador_maximo",    "Máximo multiplicador de ganancia",          "Numérica",   "Discreta"),
    "languages":            ("idiomas",                 "Idiomas disponibles",                       "Categórica", "Nominal"),
    "last_updated":         ("fecha_actualizacion",     "Fecha de última actualización",             "Temporal",   "Fecha"),
}

filas = []
for col, (es, desc, tipo, subtipo) in diccionario_columnas.items():
    filas.append({
        "Columna original": col,
        "Nombre en español": es,
        "Descripción": desc,
        "Tipo general": tipo,
        "Subtipo": subtipo,
        "Dtype pandas": str(df[col].dtype) if col in df.columns else "No encontrada",
        "Existe": col in df.columns,
    })

pd.DataFrame(filas)


**Interpretación:** El dataset contiene 20 variables que cubren aspectos técnicos, comerciales y regulatorios de los juegos. Las variables numéricas continuas (`rtp`, `min_bet`, `max_win`) son candidatas a escalado. Las categóricas nominales requieren codificación. Las booleanas pueden tratarse como indicadores 0/1.


## 3. Preparación de los Datos

Esta fase incluye:
- Detección y limpieza de valores nulos
- Eliminación de duplicados
- Segmentación por tipo de variable
- Imputación diferenciada
- Encoding de categóricas
- Escalado de numéricas


In [ ]:
# ── Limpieza previa ──────────────────────────────────────────────────────────
# Eliminar duplicados
n_antes = len(df)
df = df.drop_duplicates()
print(f"Filas eliminadas por duplicados: {n_antes - len(df)}")

# Eliminar columnas de alta cardinalidad (no aptas para modelos directamente)
cols_alta_cardinalidad = ['game', 'languages', 'country_availability', 'provider', 'last_updated']
df = df.drop(columns=[c for c in cols_alta_cardinalidad if c in df.columns])
print(f"Columnas eliminadas: {cols_alta_cardinalidad}")
print(f"Shape resultante: {df.shape}")


### 3.1 Depuración de Datos

In [ ]:
# ── Diagnóstico de nulos ANTES de imputar ────────────────────────────────────
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': nulos_pct
}).query('Nulos > 0').sort_values('Nulos', ascending=False)

print("Columnas con valores nulos:")
print(resumen_nulos if not resumen_nulos.empty else "  → No hay valores nulos")


In [ ]:
# ── Segmentación por tipo de variable ────────────────────────────────────────
num_cols  = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols  = df.select_dtypes(include=['object']).columns.tolist()
bool_cols = df.select_dtypes(include=['bool']).columns.tolist()

print(f"Numéricas  ({len(num_cols)}):  {num_cols}")
print(f"Categóricas ({len(cat_cols)}): {cat_cols}")
print(f"Booleanas  ({len(bool_cols)}): {bool_cols}")


In [ ]:
# ── Imputación diferenciada ───────────────────────────────────────────────────
# Numéricas → mediana (robusta ante outliers)
for col in num_cols:
    mediana = df[col].median()
    n_imputados = df[col].isnull().sum()
    if n_imputados > 0:
        df[col] = df[col].fillna(mediana)
        print(f"  [{col}] {n_imputados} nulos → mediana = {mediana:.4f}")

# Categóricas → moda (categoría más frecuente)
for col in cat_cols:
    moda = df[col].mode(dropna=True)
    n_imputados = df[col].isnull().sum()
    if not moda.empty and n_imputados > 0:
        df[col] = df[col].fillna(moda.iloc[0])
        print(f"  [{col}] {n_imputados} nulos → moda = '{moda.iloc[0]}'")

# Booleanas → False (criterio conservador: sin evidencia positiva)
for col in bool_cols:
    n_imputados = df[col].isnull().sum()
    if n_imputados > 0:
        df[col] = df[col].fillna(False)
        print(f"  [{col}] {n_imputados} nulos → False")

print("\n✓ Imputación completada")


In [ ]:
# ── Verificación post-imputación ─────────────────────────────────────────────
nulos_restantes = df.isnull().sum().sum()
print(f"Nulos restantes en el dataset: {nulos_restantes}")
assert nulos_restantes == 0, "¡Aún quedan nulos sin imputar!"
print("✓ Dataset libre de valores nulos")


**Interpretación de la imputación:**
- **Numéricas → mediana**: reduce el impacto de outliers y preserva la distribución central.
- **Categóricas → moda**: conserva la categoría más representativa del dataset.
- **Booleanas → False**: criterio conservador cuando no existe evidencia positiva del atributo.


### 3.2 Encoding

In [ ]:
# ── One-Hot Encoding para categóricas de baja cardinalidad ──────────────────
# Actualizar lista de categóricas tras eliminar columnas
cat_cols_actuales = df.select_dtypes(include=['object']).columns.tolist()

# Solo columnas con menos de 10 categorías únicas (evita explosión dimensional)
cat_cols_ohe = [col for col in cat_cols_actuales if df[col].nunique() < 10]
cat_cols_label = [col for col in cat_cols_actuales if df[col].nunique() >= 10]

print(f"Columnas para OHE ({len(cat_cols_ohe)}):        {cat_cols_ohe}")
print(f"Columnas descartadas/alta cardinalidad ({len(cat_cols_label)}): {cat_cols_label}")

df_encoded = pd.get_dummies(df, columns=cat_cols_ohe, drop_first=True)

# Eliminar columnas de alta cardinalidad remanentes (no codificables eficientemente)
df_encoded = df_encoded.drop(columns=cat_cols_label, errors='ignore')

print(f"\nShape tras encoding: {df_encoded.shape}")


**Interpretación:** Se aplicó One-Hot Encoding a variables categóricas con menos de 10 categorías para evitar explosión de dimensionalidad. El parámetro `drop_first=True` reduce redundancia entre columnas codificadas.


In [ ]:
# ── Escalado de variables numéricas ──────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

# Identificar columnas numéricas en el dataset codificado
num_cols_encoded = df_encoded.select_dtypes(include=['int64', 'float64']).columns.tolist()

scaler = StandardScaler()
df_encoded[num_cols_encoded] = scaler.fit_transform(df_encoded[num_cols_encoded])

print(f"Variables escaladas ({len(num_cols_encoded)}): {num_cols_encoded}")
print("\nEstadísticas post-escalado (deben ser ≈ media=0, std=1):")
print(df_encoded[num_cols_encoded].describe().loc[['mean','std']].round(4))


In [ ]:
# ── Dataset final preparado ──────────────────────────────────────────────────
df_final = df_encoded.copy()

print(f"Shape final: {df_final.shape}")
print(f"Nulos: {df_final.isnull().sum().sum()}")
df_final.head()


In [ ]:
# Verificación final
df_final.info()


## 4. Análisis Exploratorio de Datos

Se exploran distribuciones, correlaciones y relaciones entre variables clave para identificar hipótesis relevantes antes del modelado.


In [ ]:
# ── Estadísticas descriptivas ────────────────────────────────────────────────
df_final.select_dtypes(include=[np.number]).describe().T.round(3)


In [ ]:
# ── Medidas de tendencia central ─────────────────────────────────────────────
num_df = df_final.select_dtypes(include=[np.number])

print("Media:")
print(num_df.mean().round(4))
print("\nMediana:")
print(num_df.median().round(4))


In [ ]:
# ── Dispersión: desviación estándar e IQR ────────────────────────────────────
print("Desviación estándar:")
print(num_df.std().round(4))

Q1 = num_df.quantile(0.25)
Q3 = num_df.quantile(0.75)
IQR = Q3 - Q1
print("\nIQR (Rango Intercuartílico):")
print(IQR.round(4))


In [ ]:
# ── Boxplot de variables numéricas originales ────────────────────────────────
# Usamos el df original (antes de escalar) para mejor interpretabilidad visual
vars_box = ['min_bet', 'max_win', 'rtp']
vars_box_presentes = [v for v in vars_box if v in df.columns]

fig, axes = plt.subplots(1, len(vars_box_presentes), figsize=(12, 5))
for ax, col in zip(axes, vars_box_presentes):
    ax.boxplot(df[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.set_title(col)
    ax.set_ylabel('Valor')

plt.suptitle('Distribución y outliers - Variables numéricas clave', fontsize=13)
plt.tight_layout()
plt.show()


**Interpretación:** Los boxplots revelan la presencia de valores atípicos en `max_win` y `min_bet`, evidenciando juegos con características extremas. El `rtp` se muestra más concentrado y simétrico.


In [ ]:
# ── Matriz de correlación ────────────────────────────────────────────────────
corr_df = df_final.select_dtypes(include=[np.number])
corr_matrix = corr_df.corr()

plt.figure(figsize=(13, 9))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False,
            linewidths=0.3, square=True)
plt.title('Matriz de correlación entre variables numéricas', fontsize=13)
plt.tight_layout()
plt.show()

# Top 10 pares con mayor correlación absoluta
corr_pares = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .sort_values(key=lambda s: s.abs(), ascending=False)
)
print("Top 10 pares con mayor correlación absoluta:")
print(corr_pares.head(10).round(4))


**Interpretación:** Correlaciones cercanas a ±1 indican relación lineal fuerte. Pares con alta correlación entre predictores pueden indicar redundancia, lo que conviene revisar antes del modelado para evitar multicolinealidad.


## 5. Modelado y Justificación

### a. Tipo de problema
Se trata de un problema de **clasificación binaria**: determinar si un juego tiene "alto potencial" (RTP y multiplicador máximo por encima de la mediana) o no.

### b. Creación de variable objetivo


In [ ]:
# ── Variable objetivo binaria ─────────────────────────────────────────────────
# Un juego es "alto potencial" si supera la mediana en RTP Y en multiplicador máximo
df_final['target'] = (
    (df_final['rtp'] > df_final['rtp'].median()) &
    (df_final['max_multiplier'] > df_final['max_multiplier'].median())
).astype(int)

dist = df_final['target'].value_counts()
print("Distribución de clases:")
print(dist)
print(f"\nBalance: {dist[1]/len(df_final)*100:.1f}% clase 1 | {dist[0]/len(df_final)*100:.1f}% clase 0")


### c. Modelos utilizados

1. **Regresión Logística** — línea base interpretable para clasificación binaria.
2. **Árbol de Decisión** — captura relaciones no lineales y genera reglas interpretables.

**Justificación:** Ambos modelos son adecuados para clasificación con variables mixtas. La regresión logística ofrece estabilidad y la base comparativa; el árbol agrega flexibilidad no lineal. Comparar ambos permite seleccionar el de mejor desempeño empírico.


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Variables predictoras
X = df_final.drop(columns=['target']).copy()

# Codificar columnas object remanentes (si las hubiera)
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# Variable objetivo
y = df_final['target']

print(f"X shape: {X.shape}")
print(f"y distribución:\n{y.value_counts()}")


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")


## 6. Entrenamiento y Evaluación del Modelo

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

# ── Regresión Logística ───────────────────────────────────────────────────────
log_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=8000, solver="saga", tol=1e-3, random_state=42)),
])

log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)

print("=== REGRESIÓN LOGÍSTICA ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_log):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred_log):.4f}")
print()
print(classification_report(y_test, y_pred_log))


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# ── Árbol de Decisión ─────────────────────────────────────────────────────────
tree_model = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_model.fit(X_train, y_train)
y_pred_tree = tree_model.predict(X_test)

print("=== ÁRBOL DE DECISIÓN ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_tree):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred_tree):.4f}")
print()
print(classification_report(y_test, y_pred_tree))


In [ ]:
# ── Comparación de modelos ────────────────────────────────────────────────────
resumen = pd.DataFrame({
    'Modelo': ['Regresión Logística', 'Árbol de Decisión'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_tree),
    ],
    'F1 Score': [
        f1_score(y_test, y_pred_log),
        f1_score(y_test, y_pred_tree),
    ],
})
print(resumen.to_string(index=False))


**Interpretación de resultados:**
- La **Accuracy** indica el porcentaje global de predicciones correctas.
- El **F1-Score** balancea precisión y recall; es más informativo ante posible desbalance de clases.
- Si el F1 es notablemente inferior a la accuracy, puede existir sesgo hacia la clase mayoritaria y conviene revisar el balanceo.
- Si el árbol supera a la logística, ello sugiere que existen relaciones no lineales relevantes en los datos.


## 7. Despliegue (opcional)

El modelo podría integrarse en la plataforma del casino como un sistema de recomendación interno que etiqueta automáticamente cada juego al momento de su carga en el catálogo. Un operador podría usar la clasificación para:
- Destacar juegos "alto potencial" en la página de inicio.
- Enviar notificaciones personalizadas a usuarios de alto riesgo.
- Auditar el catálogo periódicamente para mantener una oferta atractiva.


In [ ]:
# ── Simulación de predicción para un nuevo juego ─────────────────────────────
# Ejemplo: usar el mejor modelo para predecir sobre los primeros 5 registros de test
muestra = X_test.iloc[:5]
preds = tree_model.predict(muestra)

print("Predicciones para 5 juegos de ejemplo:")
for i, pred in enumerate(preds):
    etiqueta = "Alto potencial" if pred == 1 else "Estándar"
    print(f"  Juego {i+1}: {etiqueta}")


## 8. Conclusiones y Recomendaciones

### Hallazgos principales
- El dataset de juegos de casino contiene variables técnicas y comerciales que permiten construir un modelo de clasificación funcional.
- La limpieza de datos resultó crítica: se detectaron e imputaron valores nulos y se eliminaron columnas de alta cardinalidad que habrían saturado el espacio de características.
- La correlación entre variables numéricas es moderada-baja en general, lo que reduce el riesgo de multicolinealidad en los modelos lineales.
- Ambos modelos (regresión logística y árbol de decisión) ofrecen una clasificación razonable; el árbol tiende a capturar mejor las interacciones no lineales.

### Limitaciones
- La variable objetivo fue construida artificialmente (mediana de RTP y multiplicador); en un contexto real debería basarse en datos de comportamiento del jugador.
- No se incorporaron variables temporales (p. ej., tendencias por año de lanzamiento).
- El dataset podría tener sesgos según los casinos incluidos en la muestra.

### Propuestas de mejora
- Incorporar datos de sesiones reales de usuarios para definir una variable objetivo más robusta.
- Explorar modelos ensemble (Random Forest, XGBoost) para mejorar el F1-Score.
- Aplicar validación cruzada k-fold para obtener estimaciones más estables del desempeño.
- Analizar importancia de variables (feature importance) para reducir dimensionalidad.

### Aplicabilidad
Este modelo es directamente aplicable en departamentos de producto y marketing de operadores de casino online, permitiéndoles tomar decisiones de catálogo y segmentación basadas en datos.

---

## Cierre metodológico (CRISP-DM)

1. **Comprensión del negocio** — se definió el problema y el objetivo analítico.
2. **Comprensión de los datos** — se evaluó estructura, tipos, nulos, dispersión y correlación.
3. **Preparación de los datos** — imputación limpia con verificación, encoding y escalado justificados.
4. **Análisis exploratorio** — estadísticas descriptivas, boxplots y matriz de correlación.
5. **Modelado** — entrenamiento y comparación de regresión logística y árbol de decisión.
6. **Evaluación** — métricas de accuracy y F1-score en conjunto de prueba.
